# Time Series Logs
Adding artificial and tool wear time stamps to the dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
BASE_DIR = Path().resolve().parent

In [2]:

#^ load featured dataset
df_feat = pd.read_csv(BASE_DIR / 'Dataset' / 'ai4i2020_features.csv')

#~ Drop index (if exist)
if 'Unnamed: 0' in df_feat.columns:
    df_feat = df_feat.drop(columns=['Unnamed: 0'])

#? Establish the Cycle_ID (to prevent products order)
df_feat['Cycle_ID'] = df_feat.index
#& sequential 1-minute Timestamp index
df_feat['Timestamp'] = pd.date_range(start='2026-01-01 00:00:00', periods=len(df_feat), freq='min')
df_feat = df_feat.set_index('Timestamp')

#todo Extract tool_cycle lifespans using delta logic
df_feat['tool_cycle'] = (df_feat['Tool wear [min]'].diff().fillna(0) < 0).cumsum()

print(f"Structure baseline set. Shape: {df_feat.shape}")
print(f"Total unique tool cycles isolated: {df_feat['tool_cycle'].nunique()}")

Structure baseline set. Shape: (10000, 58)
Total unique tool cycles isolated: 120


In [3]:
df_feat

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,...,log_strain,distance_to_twf,distance_to_hdf_temp,distance_to_hdf_rpm,distance_to_pwf_lower,distance_to_pwf_upper,distance_to_osf,risk_score,Cycle_ID,tool_cycle
Timestamp,,,,,,,,,,,,,,,,,,,,,
2026-01-01 00:00:00,1,298.1,308.6,1551,42.8,0,0,0,0,0,...,0.0000,200,1.9,171,26382.8,13617.2,12000.0,0,0,0
2026-01-01 00:01:00,0,298.2,308.7,1408,46.3,3,0,0,0,0,...,4.9409,197,1.9,28,25190.4,14809.6,10861.1,0,1,0
2026-01-01 00:02:00,0,298.1,308.5,1498,49.4,5,0,0,0,0,...,5.5134,195,1.8,118,34001.2,5998.8,10753.0,0,2,0
2026-01-01 00:03:00,0,298.2,308.6,1433,39.5,7,0,0,0,0,...,5.6258,193,1.8,53,16603.5,23396.5,10723.5,0,3,0
2026-01-01 00:04:00,0,298.2,308.7,1408,40.0,9,0,0,0,0,...,5.8889,191,1.9,28,16320.0,23680.0,10640.0,0,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-07 22:35:00,1,298.8,308.4,1604,29.5,14,0,0,0,0,...,6.0259,186,1.0,224,7318.0,32682.0,11587.0,0,9995,119
2026-01-07 22:36:00,2,298.9,308.4,1632,31.8,17,0,0,0,0,...,6.2945,183,0.9,252,11897.6,28102.4,12459.4,0,9996,119
2026-01-07 22:37:00,1,299.0,308.6,1645,33.4,22,0,0,0,0,...,6.6010,178,1.0,265,14943.0,25057.0,11265.2,0,9997,119


## Time Series Feature Engineering

Here we build temporal features across three distinct layers, each capturing a different dimension of machine behaviour over time.

In [4]:
# Signals we care about — these are the core physical and derived measurements
# these will be the main features we analyze for patterns leading to failures
core_signals = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'power', 'temp_diff']

### Layer A — Global Chronological Windows

These features roll across the entire dataset in time order, ignoring tool boundaries.

 Two stats per signal per window:

 **Rolling mean** - smoothed baseline (where is this signal sitting right now?)

 **Rolling std** - local volatility (how noisy or stable is it?)

In [5]:
for window in [5, 15]:
    for col in core_signals:
        # Rolling Mean: Tracks shifting thermal and load baselines over operational clock time
        df_feat[f'{col}_global_mean_{window}'] = df_feat[col].rolling(window=window, min_periods=1).mean()
        
        # Rolling Std: Captures micro-vibrations or ambient noise fluctuations
        df_feat[f'{col}_global_std_{window}'] = df_feat[col].rolling(window=window, min_periods=1).std().fillna(0)

print(f"Layer A done. Columns so far: {df_feat.shape[1]}")

Layer A done. Columns so far: 82


### Layer B — Intra-Tool Lifecycle Windows

Unlike Layer A, these windows reset on every tool swap — they only look within the lifespan of the current tool.

This is critical because a tool that's been running for 180 minutes behaves very differently from a fresh one at minute 5. Pooling them together would blur that signal completely.

We compute three things per signal per window:

 **Tool mean** - the signal's average *within this tool's life so far*

 **Tool std** - how much it's been fluctuating within the current tool

 **Tool deviation** - how far the current reading is from that local mean


In [6]:
# Group by tool_cycle so all rolling calculations stay within tool boundaries
roll_group = df_feat.groupby('tool_cycle')

for window in [5, 15]:
    for col in core_signals:
        # Local Mean & Std: Captures dynamics relative ONLY to the current tool's lifespan
        df_feat[f'{col}_tool_mean_{window}'] = roll_group[col].rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True).values
        df_feat[f'{col}_tool_std_{window}'] = roll_group[col].rolling(window=window, min_periods=1).std().fillna(0).reset_index(level=0, drop=True).values
        
        # Trajectory/Velocity Feature: Calculates how much the current value deviates from this tool's recent past
        df_feat[f'{col}_tool_dev_{window}'] = df_feat[col] - df_feat[f'{col}_tool_mean_{window}']

print(f"Layer B done. Columns so far: {df_feat.shape[1]}")

Layer B done. Columns so far: 118


### Layer C — Sequential Signal Memory (Lag Features)
Models can't look backwards on their own, so we manually hand them the recent past as input features. This is especially useful for catching sudden load changes right before a failure event — the lag features preserve that context.

In [7]:
for lag in [1, 2]:
    for col in ['Torque [Nm]', 'Rotational speed [rpm]', 'power']:
        df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag).bfill()

print(f"Layer C done. Columns so far: {df_feat.shape[1]}")

Layer C done. Columns so far: 124


### Reset Index and Export

The Timestamp is currently the index, which makes it invisible during export. We reset it here to bring it back as a regular column so the final CSV has a proper `Timestamp` field.

In [8]:
# Reset index to turn 'Timestamp' into a flat column for easy file exporting
df_final = df_feat.reset_index()
df_final.to_csv(BASE_DIR / 'Dataset' / 'ai4i2020_time_series.csv', index=False)

print(f"\nTime Series Generation Complete!")
print(f"Final Expanded Shape: {df_final.shape} (Expected ~125 columns)")


Time Series Generation Complete!
Final Expanded Shape: (10000, 125) (Expected ~125 columns)


In [9]:
df_feat

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,...,power_tool_dev_15,temp_diff_tool_mean_15,temp_diff_tool_std_15,temp_diff_tool_dev_15,Torque [Nm]_lag_1,Rotational speed [rpm]_lag_1,power_lag_1,Torque [Nm]_lag_2,Rotational speed [rpm]_lag_2,power_lag_2
Timestamp,,,,,,,,,,,,,,,,,,,,,
2026-01-01 00:00:00,1,298.1,308.6,1551,42.8,0,0,0,0,0,...,0.000000,10.500000,0.000000,0.000000,42.8,1551.0,66382.8,42.8,1551.0,66382.8
2026-01-01 00:01:00,0,298.2,308.7,1408,46.3,3,0,0,0,0,...,-596.200000,10.500000,0.000000,0.000000,42.8,1551.0,66382.8,42.8,1551.0,66382.8
2026-01-01 00:02:00,0,298.1,308.5,1498,49.4,5,0,0,0,0,...,5476.400000,10.466667,0.057735,-0.066667,46.3,1408.0,65190.4,42.8,1551.0,66382.8
2026-01-01 00:03:00,0,298.2,308.6,1433,39.5,7,0,0,0,0,...,-8940.975000,10.450000,0.057735,-0.050000,49.4,1498.0,74001.2,46.3,1408.0,65190.4
2026-01-01 00:04:00,0,298.2,308.7,1408,40.0,9,0,0,0,0,...,-7379.580000,10.460000,0.054772,0.040000,39.5,1433.0,56603.5,49.4,1498.0,74001.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-07 22:35:00,1,298.8,308.4,1604,29.5,14,0,0,0,0,...,-5919.885714,9.600000,0.081650,0.000000,27.9,1634.0,45588.6,47.3,1401.0,66267.3
2026-01-07 22:36:00,2,298.9,308.4,1632,31.8,17,0,0,0,0,...,-1172.750000,9.587500,0.083452,-0.087500,29.5,1604.0,47318.0,27.9,1634.0,45588.6
2026-01-07 22:37:00,1,299.0,308.6,1645,33.4,22,0,0,0,0,...,1664.577778,9.588889,0.078174,0.011111,31.8,1632.0,51897.6,29.5,1604.0,47318.0
